# 🎙️ Sugidanon — Code-Switch Hiligaynon ASR Benchmark

**Hiligaynon (Ilonggo): 9M+ speakers, near-invisible to speech tech.**
This notebook reproduces — on a fresh cloud machine, in minutes — the finding
that off-the-shelf models *catch the borrowed English/Tagalog words and miss
the Ilonggo*.

It downloads the open dataset, runs a Whisper baseline, and scores the
**switch penalty** (accuracy at the moment the language switches).

📦 Dataset: https://huggingface.co/datasets/LauelKills/sugidanon-hil-codeswitch  
💻 Code: https://github.com/Jazztinn/tinig-sa-liwanag

▶️ **Runtime → Run all.** No setup, ~3–5 min on the free CPU runtime.


## 1. Install


In [ ]:
!pip install -q openai-whisper huggingface_hub datasets
!apt-get -qq install -y ffmpeg >/dev/null


## 2. Get the code + dataset


In [ ]:
!git clone -q https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag
from huggingface_hub import snapshot_download
DS = snapshot_download('LauelKills/sugidanon-hil-codeswitch', repo_type='dataset')
print('dataset at:', DS)


## 3. Run the Whisper baseline
Whisper has no native Hiligaynon; `tl` (Tagalog) is the closest — and that is
exactly the point. Watch it nail the English/Tagalog bits and mangle the Ilonggo.


In [ ]:
!python scripts/run_whisper.py --audio-dir $DS/data/audio --out-dir /content/preds --model small --language tl


## 4. Score — overall / switch-region / monolingual WER + switch penalty


In [ ]:
!python score.py --ref $DS/data/annotations --hyp /content/preds


## 🔍 What you just saw

A **negative switch penalty** — switch-region words scored *better* than the
monolingual Hiligaynon. Why: switch regions carry the loanwords the model
already knows; the Hiligaynon matrix is what it can't handle. `tl↔en` is
near-solved (~6%), `hil↔en` is worst (~41%) — the gap scales with Hiligaynon.

That gap is the problem Sugidanon exists to measure. Swap `--model small` for
`large-v3`, or run Meta MMS, to put another model on the same benchmark.


## (Optional) Developer one-liner — `load_dataset`
Audiofolder layout, so the dataset loads in a single line. If this cell errors,
ignore it — the benchmark above is independent.


In [ ]:
from datasets import load_dataset
ds = load_dataset('LauelKills/sugidanon-hil-codeswitch', data_dir='data/audio', split='train')
print(ds)
print(ds[0]['transcript'], '|', ds[0]['switch_type'])
